# PDF Knowledge Benchmark: zero-shot vs. with document context

This notebook measures **how much an LLM already knows about the subject of a PDF**, and **how much its accuracy improves when it gets the relevant parts of the PDF as context**.

Two local open-source models run on the Colab GPU. They are loaded **one at a time** and swapped, so both fit on a T4.

| Role | Model | Job |
|---|---|---|
| Examiner | `Qwen/Qwen3-8B` | writes the questions, grades the answers |
| Student | `microsoft/phi-4` (14B) | answers the questions |

Using different model families for the two roles avoids self-preference: a model grading its own answers tends to be too lenient.

**Workflow**
1. Load the PDF and split it into chunks. Bibliography and table-of-contents chunks are not used for questions.
2. **Examiner** writes **200 question-answer pairs** from the chunks, filters and reviews each one, and saves the accepted pairs.
3. **Student, zero-shot:** answers every question without the document.
4. **Student, with context:** for each question, the 8 most relevant chunks are retrieved (embedding search) and given as context.
5. **Examiner** grades both sets of answers against the reference answers, fact by fact.
6. **Report:** accuracy for both runs, the improvement with a confidence interval, and per-question changes.

**Setup**
- Runtime → Change runtime type → **GPU** (a T4 works; L4 or A100 is faster).
- Upload the **PDF** to `/content`.
- Both models are ungated, so no Hugging Face token is needed. Adding an `HF_TOKEN` in Colab Secrets only speeds up downloads.

Every stage saves its output to disk. If the runtime disconnects, run all cells again and finished stages are loaded from disk instead of run again.

In [ ]:
!pip -q install -U transformers accelerate bitsandbytes pymupdf sentence-transformers pandas matplotlib tabulate

## 1. Configuration

In [ ]:
import os, re, glob

WORK_DIR = "/content"

EXAMINER_MODEL_ID = "Qwen/Qwen3-8B"           # writes the questions and grades the answers
STUDENT_MODEL_ID = "microsoft/phi-4"           # the model being tested; alternative: "ibm-granite/granite-3.3-8b-instruct"
EMBED_MODEL_ID = "BAAI/bge-small-en-v1.5"      # used for retrieval in the with-context run
LOAD_IN_4BIT = "auto"          # "auto" = 4-bit when the GPU has under 40 GB of memory; or True / False

NUM_QUESTIONS = 200
CHUNK_CHARS = 1500             # approximate chunk size, in characters (part of the questions file name)
MIN_CHUNK_CHARS = 400          # skip shorter chunks (title pages, fragments)
RETRIEVAL_TOP_K = 8            # chunks given to the student per question in the with-context run
REVIEW_MIN_SCORE = 4           # keep questions the examiner's review rates at least this high (1-5)

# Batch sizes; lower them if you run out of GPU memory
GEN_BATCH_SIZE = 6             # examiner writing questions
JUDGE_BATCH_SIZE = 8           # examiner grading
ANSWER_BATCH_SIZE = 8          # student, zero-shot
CONTEXT_BATCH_SIZE = 4         # student, with context (longer prompts)

pdfs = sorted(glob.glob(os.path.join(WORK_DIR, "*.pdf")))
assert pdfs, f"No PDF found in {WORK_DIR}. Upload it first."
PDF_PATH = pdfs[0]

def out(name):
    return os.path.join(WORK_DIR, name)

STUDENT_NAME = STUDENT_MODEL_ID

# File names include the question set and the student, so different runs never mix their saved results.
# Bump QA_VERSION when the question-writing prompts change, to generate a fresh set.
QA_VERSION = "v2"
QSET_TAG = f"{NUM_QUESTIONS}_c{CHUNK_CHARS}_{QA_VERSION}"
RUN_TAG = QSET_TAG + "_" + re.sub(r"[^A-Za-z0-9.]+", "-", STUDENT_NAME.split("/")[-1])
QA_PATH = out(f"qa_pairs_{QSET_TAG}.json")
QA_REJECTED_PATH = out(f"qa_rejected_{QSET_TAG}.json")
ANS_PATHS = {l: out(f"answers_{l}_{RUN_TAG}.json") for l in ("zero_shot", "with_context")}
JUDGE_PATHS = {l: out(f"judgments_{l}_{RUN_TAG}.json") for l in ("zero_shot", "with_context")}
RESULTS_CSV = out(f"benchmark_results_{RUN_TAG}.csv")
REPORT_MD = out(f"benchmark_report_{RUN_TAG}.md")
CHART_PNG = out(f"benchmark_chart_{RUN_TAG}.png")

print("PDF:", PDF_PATH, "| Student:", STUDENT_NAME, "| Questions file:", QA_PATH)

## 2. Load the PDF and split it into chunks

In [ ]:
import re, json, math, random, collections, fitz  # PyMuPDF

def load_pdf_text(path):
    doc = fitz.open(path)
    pages = []
    for page in doc:
        t = page.get_text("text")
        t = re.sub(r"-\n(\w)", r"\1", t)        # rejoin words hyphenated across lines
        t = re.sub(r"[ \t]+", " ", t)
        pages.append(t.strip())
    return pages

def chunk_text(pages, chunk_chars=CHUNK_CHARS, min_chars=MIN_CHUNK_CHARS):
    # Split into paragraphs, then pack paragraphs into chunks of about chunk_chars
    paras = []
    for p_no, page in enumerate(pages, 1):
        for para in re.split(r"\n\s*\n", page):
            para = " ".join(para.split())
            if para:
                paras.append((p_no, para))
    chunks, buf, start_page = [], "", None
    for p_no, para in paras:
        if start_page is None:
            start_page = p_no
        if len(buf) + len(para) > chunk_chars and len(buf) >= min_chars:
            chunks.append({"text": buf.strip(), "pages": f"{start_page}-{p_no}"})
            buf, start_page = "", p_no
        buf += para + "\n"
    if len(buf.strip()) >= min_chars:
        chunks.append({"text": buf.strip(), "pages": f"{start_page}-{paras[-1][0]}"})
    for i, c in enumerate(chunks):
        c["chunk_id"] = i
    return chunks

# Bibliography and table-of-contents chunks are kept for retrieval but never used to write questions
CITATION = re.compile(r"\(\s*(19|20)\d{2}[a-z]?\s*\)|\b(19|20)\d{2}[a-z]?\)\.|\bpp\.|\bVol\.|\bJournal\b|\bPress\b|\bEds?\.", re.I)

def skip_reason(text):
    if 1000 * len(CITATION.findall(text)) / len(text) >= 4:   # body text measured under 3 citations per 1,000 characters
        return "bibliography"
    if len(re.findall(r"\.{5,}", text)) >= 3:                # dotted leader lines
        return "table of contents"
    return None

pages = load_pdf_text(PDF_PATH)
chunks = chunk_text(pages)
for c in chunks:
    c["skip"] = skip_reason(c["text"])
qa_chunks = [c for c in chunks if not c["skip"]]
print(f"{len(pages)} pages, {sum(len(p) for p in pages):,} characters, {len(chunks)} chunks "
      f"({len(qa_chunks)} used for questions)")
for c in chunks:
    if c["skip"]:
        print(f"  skipped chunk {c['chunk_id']} (p. {c['pages']}, {c['skip']}): {' '.join(c['text'].split())[:80]}...")
print("\nSample chunk:\n", chunks[len(chunks) // 2]["text"][:800])

## 3. Model loading helpers

Only one model is on the GPU at a time. `load_llm` frees the previous model before loading the next one.

In [ ]:
import gc, torch
from transformers import AutoTokenizer, AutoModelForCausalLM, BitsAndBytesConfig

assert torch.cuda.is_available(), "No GPU. Runtime → Change runtime type → GPU"
gpu_mem_gb = torch.cuda.get_device_properties(0).total_memory / 1e9
use_4bit = (gpu_mem_gb < 40) if LOAD_IN_4BIT == "auto" else bool(LOAD_IN_4BIT)
dtype = torch.bfloat16 if torch.cuda.is_bf16_supported() else torch.float16
print(f"GPU: {torch.cuda.get_device_name(0)} ({gpu_mem_gb:.1f} GB) | 4-bit: {use_4bit} | dtype: {dtype}")

llm = {"id": None, "model": None, "tokenizer": None}

def free_gpu():
    llm.update({"id": None, "model": None, "tokenizer": None})
    gc.collect()
    torch.cuda.empty_cache()

def load_llm(model_id):
    if llm["id"] == model_id:
        return
    free_gpu()
    print(f"Loading {model_id} ...")
    tok = AutoTokenizer.from_pretrained(model_id)
    tok.padding_side = "left"
    if tok.pad_token is None:
        tok.pad_token = tok.eos_token
    quant = BitsAndBytesConfig(load_in_4bit=True, bnb_4bit_quant_type="nf4",
                               bnb_4bit_compute_dtype=dtype, bnb_4bit_use_double_quant=True) if use_4bit else None
    mdl = AutoModelForCausalLM.from_pretrained(model_id, device_map="auto", dtype=dtype, quantization_config=quant)
    mdl.eval()
    llm.update({"id": model_id, "model": mdl, "tokenizer": tok})
    print(f"Loaded. GPU memory in use: {torch.cuda.memory_allocated() / 1e9:.1f} GB")

def local_chat(prompts, max_new_tokens=512, batch_size=8, system=None, desc=None):
    """Greedy, batched generation with the currently loaded model. Returns one string per prompt."""
    from tqdm.auto import tqdm
    tok, mdl = llm["tokenizer"], llm["model"]
    outs = []
    for i in tqdm(range(0, len(prompts), batch_size), desc=desc, disable=desc is None):
        texts = []
        for p in prompts[i:i + batch_size]:
            msgs = ([{"role": "system", "content": system}] if system else []) + [{"role": "user", "content": p}]
            # enable_thinking=False turns off Qwen3's reasoning mode; other chat templates ignore it
            texts.append(tok.apply_chat_template(msgs, tokenize=False, add_generation_prompt=True, enable_thinking=False))
        enc = tok(texts, return_tensors="pt", padding=True).to(mdl.device)
        with torch.no_grad():
            gen = mdl.generate(**enc, max_new_tokens=max_new_tokens, do_sample=False,
                               temperature=None, top_p=None, top_k=None, pad_token_id=tok.pad_token_id)
        for g in gen[:, enc["input_ids"].shape[1]:]:
            t = tok.decode(g, skip_special_tokens=True)
            outs.append(re.sub(r"<think>.*?</think>", "", t, flags=re.S).strip())
    return outs

def extract_json(text):
    """Find and parse the first JSON array or object in the model output."""
    text = re.sub(r"```(?:json)?", "", text)
    for open_c, close_c in (("[", "]"), ("{", "}")):
        s, e = text.find(open_c), text.rfind(close_c)
        if s != -1 and e > s:
            try:
                return json.loads(text[s:e + 1])
            except json.JSONDecodeError:
                continue
    return None

def load_json(path, default):
    return json.load(open(path)) if os.path.exists(path) else default

def save_json(obj, path):
    json.dump(obj, open(path, "w"), indent=2, ensure_ascii=False)

## 4. Examiner writes and reviews 200 question-answer pairs (cached)

Each question must make sense on its own, without the document, so the zero-shot run is fair. It must also have a single checkable answer that is knowledge about the subject, not a detail only this document would know. Quality is checked in three steps:
1. **Prompt:** rules plus good and bad examples from an unrelated subject.
2. **Word filter:** drops questions that mention the source ("the passage", "the report", "mentioned", "cited", ...) and open-ended questions with many valid answers ("What is one strategy for...", "examples of ...").
3. **Review:** the examiner rates each remaining question from 1 to 5 against its passage, using a rubric that names the common failure modes. A question is kept only if it scores at least `REVIEW_MIN_SCORE`, has a single expected answer, and could be answered by an expert who has not seen the document.

Rejected questions are saved to `qa_rejected_*.json` with the reason, so you can inspect them. If the questions file already exists, it is loaded instead of generated again. Delete it, or change `QA_VERSION`, to generate a new set.

In [ ]:
QA_PROMPT = """You are writing questions for an exam that tests how well someone knows a subject.
The passage below comes from a document about that subject. The person taking the exam will NOT see the passage,
so every question must make complete sense on its own.

Write up to {n} question-answer pairs from the passage.

Good questions:
- ask about the subject itself: facts, definitions, causes, effects, criteria, recommended practices, and the reasons behind them
- name the topic explicitly instead of relying on context (write "patients with diabetes", not "these patients")
- have exactly ONE correct answer, which the passage states and a grader can check

Never write questions that:
- refer to the source: "the passage", "the text", "the report", "the document", "the author(s)", "this study", "mentioned", "described", "outlined", "the guide"
- ask about the document itself or its bibliography: its title, purpose, focus, audience, publisher, staff, cited works, or their authors
- ask for "one" example, way, strategy, method, or tool when several valid answers exist ("What is one way to ...?", "What are examples of ...?").
  Instead, ask for the complete list the passage gives ("What are the three ...?"), or ask about one specific named item ("What is the purpose of X?")
- depend on figures likely to change over time (current counts, prevalence rates, officials, "upcoming" publications)
- are vague or opinion-based, or can be answered with only yes or no

Examples from an unrelated subject:
BAD:  "What is the main focus of the passage?"                     (about the document)
BAD:  "According to the author, what helps plants grow faster?"     (refers to the source)
BAD:  "What is one way to help houseplants grow?"                   (many valid answers)
BAD:  "Why is photosynthesis important?"                            (vague)
GOOD: "Which pigment absorbs most of the light energy used in photosynthesis?" -> "Chlorophyll, mainly chlorophyll a."
GOOD: "What are the two stages of photosynthesis?" -> "The light-dependent reactions and the Calvin cycle."
GOOD: "Why do leaves turn yellow and orange in autumn?" -> "Chlorophyll breaks down, revealing carotenoid pigments that were already present in the leaf."

Answers: 1-2 sentences, specific, and fully supported by the passage. Include the key details the passage gives (numbers, names, list items).
Each question must test a different fact.{avoid}
If the passage has little substantive content (title page, ordering information, acknowledgements), write fewer pairs or return [].

Return ONLY a JSON array, with no other text:
[{{"question": "...", "answer": "..."}}]

PASSAGE:
\"\"\"
{chunk}
\"\"\""""

REVIEW_PROMPT = """You are a demanding reviewer of exam questions. Most draft questions have at least one flaw; be critical.
The exam tests knowledge of the SUBJECT. The person taking it will NOT see the passage.

PASSAGE:
\"\"\"
{chunk}
\"\"\"

QUESTION: {question}
ANSWER: {answer}

Answer these checks:
- "single_answer": true only if there is ONE expected answer. False if the question asks for "one"/"an example"/"a way" of something
  and many different answers would be valid, or if it is vague.
- "expert_answerable": true only if a well-informed expert on the subject who has never seen this document could answer it correctly.
  False if it asks about the document itself, its bibliography, its staff, its own particular examples or wording,
  or about a figure likely to have changed since it was written.

Then rate the question from 1 to 5:
5 = excellent: standalone, about the subject, one clear answer, fully supported by the passage
4 = good: minor wording issues only
3 = flawed: somewhat open-ended or partly depends on this document's particular choices
2 = poor: many valid answers, about the document, or time-sensitive
1 = unusable: refers to the source, the answer is not supported by the passage, or it makes no sense

Return ONLY JSON: {{"reason": "<one sentence naming the main flaw, or 'none'>", "single_answer": true, "expert_answerable": true, "score": 4}}"""

# Questions that match these patterns fail immediately, before the review step
SOURCE_REF = re.compile(r"\b(passages?|excerpts?|the text|this text|the report|this report|the document|this document|"
                        r"the authors?|this study|this chapter|this section|the article|this article|the guide|this guide|"
                        r"mentioned|outlined|discussed|cited|journal|published|publications?|title of|book (titled|by|written))\b", re.I)
OPEN_ENDED = re.compile(r"\b(one|an?) (example|way|strategy|method|technique|tool|approach|activity|tip|idea|thing)\b|"
                        r"\bexamples? of\b|\bone of the\b", re.I)

def filter_reason(q):
    if SOURCE_REF.search(q):
        return "filter: mentions the source or the bibliography"
    if OPEN_ENDED.search(q):
        return "filter: open-ended (many valid answers)"
    return None

def norm_q(q):
    return re.sub(r"[^a-z0-9 ]", "", q.lower()).strip()

def is_dup(q, existing_norms):
    from difflib import SequenceMatcher
    nq = norm_q(q)
    return any(nq == e or SequenceMatcher(None, nq, e).ratio() > 0.85 for e in existing_norms)

def review(cands):
    prompts = [REVIEW_PROMPT.format(chunk=chunks[c["chunk_id"]]["text"], question=c["question"], answer=c["answer"])
               for c in cands]
    outs = local_chat(prompts, max_new_tokens=150, batch_size=JUDGE_BATCH_SIZE, desc="Reviewing")
    ok, bad = [], []
    for c, o in zip(cands, outs):
        r = extract_json(o)
        if not isinstance(r, dict):
            bad.append({**c, "review": "unparseable review: " + o[:200]})
            continue
        try:
            score = int(r.get("score", 0))
        except (TypeError, ValueError):
            score = 0
        passed = score >= REVIEW_MIN_SCORE and r.get("single_answer") is True and r.get("expert_answerable") is True
        (ok if passed else bad).append({**c, "review": r, "review_score": score})
    return ok, bad

def generate_qa(source_chunks, target):
    per_chunk = max(2, min(12, math.ceil(target * 1.8 / len(source_chunks))))   # overgenerate; filter and review reject some
    accepted = {c["chunk_id"]: [] for c in source_chunks}
    seen = {c["chunk_id"]: [] for c in source_chunks}   # every question written per chunk, accepted or not
    rejected, norms = [], []

    def run_pass(chunk_list, n, desc):
        prompts = []
        for c in chunk_list:
            prev = seen[c["chunk_id"]]
            avoid = ("\nDo NOT repeat or rephrase these existing questions:\n" + "\n".join(f"  * {q}" for q in prev)) if prev else ""
            prompts.append(QA_PROMPT.format(n=n, chunk=c["text"], avoid=avoid))
        outs = local_chat(prompts, max_new_tokens=200 * n + 100, batch_size=GEN_BATCH_SIZE, desc=desc)
        cands, n_filtered = [], 0
        for c, o in zip(chunk_list, outs):
            items = extract_json(o)
            if not isinstance(items, list):
                continue
            for it in items:
                if not isinstance(it, dict):
                    continue
                q, a = str(it.get("question", "")).strip(), str(it.get("answer", "")).strip()
                if len(q) < 15 or len(a) < 2 or is_dup(q, norms):
                    continue
                norms.append(norm_q(q)); seen[c["chunk_id"]].append(q)
                qa = {"question": q, "answer": a, "chunk_id": c["chunk_id"], "pages": c["pages"]}
                why = filter_reason(q)
                if why:
                    rejected.append({**qa, "review": why}); n_filtered += 1
                else:
                    cands.append(qa)
        ok, bad = review(cands) if cands else ([], [])
        for qa in ok:
            accepted[qa["chunk_id"]].append(qa)
        rejected.extend(bad)
        print(f"  {n_filtered} dropped by the filter, {len(cands)} reviewed, {len(ok)} accepted "
              f"| total accepted: {sum(map(len, accepted.values()))}")

    print(f"Asking for {per_chunk} question(s) per chunk across {len(source_chunks)} chunks")
    run_pass(source_chunks, per_chunk, "Writing questions")
    for extra in range(4):  # top up the chunks with the fewest accepted questions
        if sum(map(len, accepted.values())) >= target:
            break
        needy = sorted(source_chunks, key=lambda c: len(accepted[c["chunk_id"]]))[: max(1, len(source_chunks) // 2)]
        run_pass(needy, 4, f"Top-up pass {extra + 1}")

    # Pick round-robin across chunks so the whole document is covered evenly, best-rated questions first
    pools = [sorted(v, key=lambda qa: -qa["review_score"]) for v in accepted.values() if v]
    selected = []
    while len(selected) < target and any(pools):
        for pool in pools:
            if pool and len(selected) < target:
                selected.append(pool.pop(0))
    for i, qa in enumerate(selected):
        qa["id"] = i
    return selected, rejected

if os.path.exists(QA_PATH):
    qa_pairs = load_json(QA_PATH, [])
    print(f"Loaded {len(qa_pairs)} pairs from {QA_PATH}")
else:
    load_llm(EXAMINER_MODEL_ID)
    qa_pairs, qa_rejected = generate_qa(qa_chunks, NUM_QUESTIONS)
    save_json(qa_pairs, QA_PATH)
    save_json(qa_rejected, QA_REJECTED_PATH)
    print(f"Saved {len(qa_pairs)} pairs to {QA_PATH} ({len(qa_rejected)} rejected questions in {QA_REJECTED_PATH})")
    reasons = collections.Counter(r["review"] if isinstance(r["review"], str) else f"review score {r['review_score']}"
                                  for r in qa_rejected)
    print("Rejection reasons:", dict(reasons.most_common()))
    print("\nSample of rejected questions:")
    for r in random.Random(0).sample(qa_rejected, min(10, len(qa_rejected))):
        why = r["review"] if isinstance(r["review"], str) else f"score {r['review_score']}: {r['review'].get('reason', '')}"
        print(f"  ✗ {r['question']}  [{why}]")

if len(qa_pairs) < NUM_QUESTIONS:
    print(f"⚠️ Only {len(qa_pairs)} pairs (target {NUM_QUESTIONS}). The document may be too short for that many "
          f"distinct questions; try a smaller CHUNK_CHARS or a lower NUM_QUESTIONS.")

print("\nRandom sample of accepted questions:")
for qa in random.Random(1).sample(qa_pairs, min(10, len(qa_pairs))):
    print(f"\nQ{qa['id']} (p. {qa['pages']}): {qa['question']}\nA: {qa['answer']}")

## 5. Retrieval for the with-context run

Each question is matched to the `RETRIEVAL_TOP_K` most similar chunks with a small embedding model. The **hit rate** is the share of questions whose source chunk (the one the question was written from) is among the retrieved chunks. A low hit rate limits how much the context can help.

In [ ]:
from sentence_transformers import SentenceTransformer

assert all(qa["chunk_id"] < len(chunks) for qa in qa_pairs), \
    f"{QA_PATH} was made with different chunk settings. Delete it or restore CHUNK_CHARS."

embedder = SentenceTransformer(EMBED_MODEL_ID, device="cuda")
chunk_emb = embedder.encode([c["text"] for c in chunks], normalize_embeddings=True, convert_to_tensor=True)
q_emb = embedder.encode(["Represent this sentence for searching relevant passages: " + qa["question"] for qa in qa_pairs],
                        normalize_embeddings=True, convert_to_tensor=True)
top_idx = (q_emb @ chunk_emb.T).topk(min(RETRIEVAL_TOP_K, len(chunks)), dim=1).indices.tolist()
retrieved = {qa["id"]: idx for qa, idx in zip(qa_pairs, top_idx)}
del embedder, chunk_emb, q_emb
free_gpu()

hits = [qa["chunk_id"] in retrieved[qa["id"]] for qa in qa_pairs]
print(f"Retrieval hit rate (source chunk in top {RETRIEVAL_TOP_K}): {100 * sum(hits) / len(hits):.1f}%")

## 6. Student answers (zero-shot, then with context)

In [ ]:
ANSWER_INSTRUCTIONS = "Answer the question accurately and concisely in 1-3 sentences. Give the answer directly."

def zero_shot_prompt(qa):
    return f"{ANSWER_INSTRUCTIONS}\n\nQuestion: {qa['question']}"

def context_prompt(qa):
    excerpts = "\n\n".join(f"[Excerpt {k + 1}]\n{chunks[i]['text']}" for k, i in enumerate(retrieved[qa["id"]]))
    return (f"Use the document excerpts below to answer. {ANSWER_INSTRUCTIONS}\n\n"
            f"{excerpts}\n\nQuestion: {qa['question']}")

def run_local_student(label, make_prompt, batch_size):
    path = ANS_PATHS[label]
    answers = load_json(path, {})
    todo = [qa for qa in qa_pairs if not answers.get(str(qa["id"]), "").strip()]
    print(f"[{label}] {len(qa_pairs) - len(todo)} already answered, {len(todo)} to go")
    if not todo:
        return answers
    load_llm(STUDENT_MODEL_ID)
    step = batch_size * 5   # save to disk every few batches
    for i in range(0, len(todo), step):
        part = todo[i:i + step]
        outs = local_chat([make_prompt(qa) for qa in part], max_new_tokens=200, batch_size=batch_size,
                          desc=f"{label} {i + len(part)}/{len(todo)}")
        for qa, o in zip(part, outs):
            answers[str(qa["id"])] = o
        save_json(answers, path)
    return answers

answers = {"zero_shot": run_local_student("zero_shot", zero_shot_prompt, ANSWER_BATCH_SIZE),
           "with_context": run_local_student("with_context", context_prompt, CONTEXT_BATCH_SIZE)}
free_gpu()
qa0 = qa_pairs[0]
print("\nQ:", qa0["question"], "\nReference:", qa0["answer"])
print("Zero-shot:", answers["zero_shot"][str(qa0["id"])])
print("With context:", answers["with_context"][str(qa0["id"])])

## 7. Examiner grades the answers (cached)

The examiner grades fact by fact. It lists the key facts in the reference answer, marks which ones the candidate answer covers, and notes whether the candidate contradicts the reference. The verdict follows from those checks:
- **correct** (1.0): covers every key fact and contradicts none
- **partially_correct** (0.5): covers some key facts and contradicts none
- **incorrect** (0.0): covers no key fact, or contradicts the reference

Extra information is never penalized unless it contradicts the reference.

In [ ]:
import pandas as pd

JUDGE_PROMPT = """You are grading an exam answer against a reference answer.

QUESTION: {question}
REFERENCE: {reference}
CANDIDATE: {candidate}

Steps:
1. List the key facts in the REFERENCE: the 1-4 essential pieces of information a correct answer must contain.
   Leave out wording, examples given only as illustration, and filler.
2. For each key fact, decide whether the CANDIDATE conveys it. Paraphrases, synonyms, and more specific or more
   general phrasings that mean the same thing count as covered.
3. Decide whether the CANDIDATE directly contradicts the REFERENCE (states something incompatible with it).

Rules:
- Extra information in the CANDIDATE is NEVER a problem unless it contradicts the REFERENCE.
- Longer answers are not penalized. Judge only coverage of the key facts and contradictions.
- A refusal or "the information is not available" covers no facts.

Return ONLY JSON:
{{"key_facts": ["...", "..."], "covered": [true, false], "contradicts": false, "reason": "<one short sentence>"}}"""

SCORES = {"correct": 1.0, "partially_correct": 0.5, "incorrect": 0.0}
LABELS = ["zero_shot", "with_context"]

def parse_verdict(text):
    """The verdict is computed from the fact checks, so the rules above are applied the same way every time."""
    obj = extract_json(text)
    if isinstance(obj, dict) and isinstance(obj.get("covered"), list) and obj["covered"]:
        covered = [c is True for c in obj["covered"]]
        reason = str(obj.get("reason", ""))
        facts = obj.get("key_facts") or []
        detail = "; ".join(f"{'✓' if c else '✗'} {f}" for f, c in zip(facts, covered))
        if obj.get("contradicts") is True or not any(covered):
            verdict = "incorrect"
        else:
            verdict = "correct" if all(covered) else "partially_correct"
        return verdict, f"{reason} [{detail}]"
    return "incorrect", "UNPARSEABLE: " + text[:200]

def judge(label):
    path = JUDGE_PATHS[label]
    judgments = load_json(path, {})
    todo = []
    for qa in qa_pairs:
        key, cand = str(qa["id"]), answers[label].get(str(qa["id"]), "")
        # re-grade when the answer changed since it was last graded
        if judgments.get(key, {}).get("answer") == cand:
            continue
        if not cand or cand.startswith("[ERROR]"):
            judgments[key] = {"answer": cand, "verdict": "error", "reason": cand[:200]}
        else:
            todo.append(qa)
    print(f"[{label}] {len(todo)} answers to grade")
    if todo:
        load_llm(EXAMINER_MODEL_ID)
        prompts = [JUDGE_PROMPT.format(question=qa["question"], reference=qa["answer"],
                                       candidate=answers[label][str(qa["id"])]) for qa in todo]
        for qa, o in zip(todo, local_chat(prompts, max_new_tokens=300, batch_size=JUDGE_BATCH_SIZE, desc=f"Grading {label}")):
            v, r = parse_verdict(o)
            judgments[str(qa["id"])] = {"answer": answers[label][str(qa["id"])], "verdict": v, "reason": r}
    save_json(judgments, path)
    return judgments

judgments = {label: judge(label) for label in LABELS}

rows = []
for qa, hit in zip(qa_pairs, hits):
    row = {"id": qa["id"], "pages": qa["pages"], "question": qa["question"], "reference": qa["answer"], "retrieval_hit": hit}
    for label in LABELS:
        j = judgments[label][str(qa["id"])]
        row.update({f"answer_{label}": j["answer"], f"verdict_{label}": j["verdict"],
                    f"score_{label}": SCORES.get(j["verdict"], 0.0), f"reason_{label}": j["reason"]})
    rows.append(row)
df = pd.DataFrame(rows)
df["delta"] = df.score_with_context - df.score_zero_shot
df.to_csv(RESULTS_CSV, index=False)
print("Saved", RESULTS_CSV)

## 8. Report

In [ ]:
import numpy as np
import matplotlib.pyplot as plt

def summarize(label):
    v = df[f"verdict_{label}"]
    n = len(df)
    s = {"run": label, "n": n,
         "correct": int((v == "correct").sum()),
         "partial": int((v == "partially_correct").sum()),
         "incorrect": int((v == "incorrect").sum()),
         "errors": int((v == "error").sum())}
    s["strict_acc_%"] = round(100 * s["correct"] / n, 1)                 # correct only
    s["lenient_score_%"] = round(100 * df[f"score_{label}"].mean(), 1)   # correct + 0.5 * partial
    return s

summary = pd.DataFrame([summarize(l) for l in LABELS]).set_index("run")
display(summary)

# 95% bootstrap confidence interval for the improvement (paired: same questions in both runs)
rng = np.random.default_rng(0)
d = df.delta.to_numpy()
boot = rng.choice(d, size=(5000, len(d)), replace=True).mean(axis=1) * 100
ci_lo, ci_hi = np.percentile(boot, [2.5, 97.5])

zs, wc = summary.loc["zero_shot"], summary.loc["with_context"]
improved, regressed = int((df.delta > 0).sum()), int((df.delta < 0).sum())
print(f"\nStudent: {STUDENT_NAME} | Examiner: {EXAMINER_MODEL_ID}")
print(f"Strict accuracy:  {zs['strict_acc_%']}%  →  {wc['strict_acc_%']}%   ({wc['strict_acc_%'] - zs['strict_acc_%']:+.1f} points)")
print(f"Lenient score:    {zs['lenient_score_%']}%  →  {wc['lenient_score_%']}%   "
      f"({wc['lenient_score_%'] - zs['lenient_score_%']:+.1f} points, 95% CI {ci_lo:+.1f} to {ci_hi:+.1f})")
print(f"Improved with context: {improved} | got worse: {regressed} | unchanged: {len(df) - improved - regressed}")

by_hit = df.groupby("retrieval_hit")[["score_zero_shot", "score_with_context"]].mean().mul(100).round(1)
by_hit["n"] = df.groupby("retrieval_hit").size()
by_hit.index = by_hit.index.map({True: "source chunk retrieved", False: "source chunk missed"})
print("\nLenient score by retrieval outcome:")
display(by_hit)

# Chart
order = ["correct", "partially_correct", "incorrect", "error"]
colors = {"correct": "#2e7d32", "partially_correct": "#f9a825", "incorrect": "#c62828", "error": "#9e9e9e"}
counts = pd.DataFrame({l: df[f"verdict_{l}"].value_counts().reindex(order, fill_value=0) for l in LABELS}).T
fig, axes = plt.subplots(1, 2, figsize=(12, 4))
counts.plot(kind="barh", stacked=True, color=[colors[c] for c in order], ax=axes[0])
axes[0].set_title(f"{STUDENT_NAME}: verdicts (n={len(df)})"); axes[0].set_xlabel("questions")
axes[0].legend(loc="lower right", fontsize=8)
summary[["strict_acc_%", "lenient_score_%"]].plot(kind="bar", ax=axes[1], color=["#1565c0", "#90caf9"], rot=0)
axes[1].set_ylim(0, 100); axes[1].set_title("Accuracy (%)")
for c in axes[1].containers:
    axes[1].bar_label(c, fmt="%.1f")
plt.tight_layout(); plt.savefig(CHART_PNG, dpi=120); plt.show()

# Verdict transitions: rows = zero-shot, columns = with context
transitions = pd.crosstab(df.verdict_zero_shot, df.verdict_with_context, margins=True)
display(transitions)

### Questions that changed

In [ ]:
pd.set_option("display.max_colwidth", 200)
cols = ["question", "reference", "answer_zero_shot", "verdict_zero_shot", "answer_with_context", "verdict_with_context"]
print("Improved with context:")
display(df[df.delta > 0][cols].head(15))
print("Got worse with context (also check the examiner's reasoning here):")
display(df[df.delta < 0][cols + ["retrieval_hit", "reason_with_context"]])

In [ ]:
# Save a markdown report
context_desc = f"top {RETRIEVAL_TOP_K} retrieved chunks ({EMBED_MODEL_ID}), hit rate {100 * sum(hits) / len(hits):.1f}%"
report = f"""# PDF Knowledge Benchmark

- **Document:** {os.path.basename(PDF_PATH)} ({len(pages)} pages, {len(chunks)} chunks)
- **Student (answering):** {STUDENT_NAME}
- **Examiner (question writing and grading):** {EXAMINER_MODEL_ID}
- **With-context run:** {context_desc}
- **Questions:** {len(df)}

## Summary

{summary.to_markdown()}

- Strict accuracy change: **{wc['strict_acc_%'] - zs['strict_acc_%']:+.1f} points**
- Lenient score change: **{wc['lenient_score_%'] - zs['lenient_score_%']:+.1f} points** (95% CI {ci_lo:+.1f} to {ci_hi:+.1f})
- Improved: {improved} | Got worse: {regressed} | Unchanged: {len(df) - improved - regressed}

## Verdict transitions (rows: zero-shot, columns: with context)

{transitions.to_markdown()}
"""
open(REPORT_MD, "w").write(report)
print(report)

## 9. Download the results

Files in `/content` are deleted when the runtime resets. Download them here, or copy them to Google Drive.

In [ ]:
import shutil
bundle_dir = out("benchmark_outputs")
os.makedirs(bundle_dir, exist_ok=True)
for f in [QA_PATH, QA_REJECTED_PATH, *ANS_PATHS.values(), *JUDGE_PATHS.values(), RESULTS_CSV, REPORT_MD, CHART_PNG]:
    if os.path.exists(f):
        shutil.copy(f, bundle_dir)
shutil.make_archive(bundle_dir, "zip", bundle_dir)
try:
    from google.colab import files
    files.download(bundle_dir + ".zip")
except Exception:
    print("Saved:", bundle_dir + ".zip")